# 9-bob. 9-BOB. TARMOQ HUJUMLARINI ANIQLASH: SI ASOSIDAGI BOSTIRIB KIRISHNI ANIQLASH TIZIMLARI
## Amaliy misol: paketlardan oqimlar va xost xususiyatlariga

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/09-bob/9-1_paketlardan-oqimlar-va-xost-xususiyatlariga.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 9-bob.*

Quyidagi dasturda bir daqiqalik tarmoq trafigi sun'iy ravishda yaratiladi. Unda uchta manba bor: saytlarni ko'rayotgan oddiy foydalanuvchi, serverning portlarini skanerlayotgan kompyuter va SSH xizmatiga parolni tanlab kirishga urinayotgan kompyuter. Paketlar avval beshlik bo'yicha oqimlarga birlashtiriladi, so'ngra oqimlar har bir manba bo'yicha jamlanadi. Misolda IP manzillar sifatida hujjatlar va o'quv misollari uchun maxsus ajratilgan manzillar ishlatilgan. Dastur amaliy mashg'ulotning A9 mavzusiga mos keladi.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import random
import pandas as pd

tasodif = random.Random(3)
keyingi_port = [50000]   # har yangi ulanish uchun navbatdagi manba porti
paketlar = []   # (vaqt, manba, manba porti, manzil, manzil porti, hajm, bayroq)

def ulanish(t, manba, manzil, port, soni, hajmlar, bayroq="ACK", oraliq=0.05):
    """Bitta ulanishning paketlari; manba porti har ulanishda yangi bo'ladi."""
    keyingi_port[0] += 1
    mport = keyingi_port[0]
    for k in range(soni):
        paketlar.append((t + k * oraliq, manba, mport, manzil, port, tasodif.randint(*hajmlar), bayroq))

In [2]:
# 1. Oddiy foydalanuvchi: bir nechta saytga HTTPS so'rovlari
for _ in range(40):
    server = tasodif.choice(["203.0.113.10", "198.51.100.20", "192.0.2.30"])
    ulanish(tasodif.uniform(0, 60), "10.0.0.15", server, 443, tasodif.randint(6, 20), (200, 1500))

In [3]:
# 2. Portlarni skanerlash: bitta serverning 500 ta portiga bittadan SYN paketi
for port in range(1, 501):
    ulanish(10 + port * 0.02, "10.0.0.66", "10.0.0.5", port, 1, (60, 60), "SYN")

In [4]:
# 3. Parolni tanlash: SSH portiga 120 ta qisqa ulanish
for i in range(120):
    ulanish(5 + i * 0.4, "10.0.0.77", "10.0.0.5", 22, 4, (80, 200), oraliq=0.01)

df = pd.DataFrame(paketlar, columns=["vaqt", "manba", "mport", "manzil", "port", "hajm", "bayroq"])

# Paketlardan oqimlarga: beshta maydon bir xil bo'lgan paketlar bitta oqim
oqimlar = df.groupby(["manba", "mport", "manzil", "port"]).agg(
    paketlar=("hajm", "size"), baytlar=("hajm", "sum"),
    davomiylik=("vaqt", lambda v: v.max() - v.min())).reset_index()
print(f"Paketlar soni: {len(df)}, oqimlar soni: {len(oqimlar)}\n")

# Oqimlardan har bir manba bo'yicha 60 soniyalik oyna xususiyatlari
xostlar = oqimlar.groupby("manba").agg(
    oqimlar=("port", "size"),
    noyob_portlar=("port", "nunique"),
    noyob_manzillar=("manzil", "nunique"),
    ortacha_paket=("paketlar", "mean"),
    ortacha_bayt=("baytlar", "mean"),
    ortacha_davom=("davomiylik", "mean"))
xostlar["syn_ulushi"] = df.assign(syn=df.bayroq.eq("SYN")).groupby("manba").syn.mean()
print(xostlar.round(2).to_string())

Paketlar soni: 1497, oqimlar soni: 660

           oqimlar  noyob_portlar  noyob_manzillar  ortacha_paket  ortacha_bayt  ortacha_davom  syn_ulushi
manba                                                                                                     
10.0.0.15       40              1                3          12.92      11140.72           0.60         0.0
10.0.0.66      500            500                1           1.00         60.00           0.00         1.0
10.0.0.77      120              1                1           4.00        554.09           0.03         0.0


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
Paketlar soni: 1497, oqimlar soni: 660

           oqimlar  noyob_portlar  noyob_manzillar  ortacha_paket  ortacha_bayt  ortacha_davom  syn_ulushi
manba
10.0.0.15       40              1                3          12.92      11140.72           0.60         0.0
10.0.0.66      500            500                1           1.00         60.00           0.00         1.0
10.0.0.77      120              1                1           4.00        554.09           0.03         0.0
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 9-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.